# Notebook 01 — Generating Sample Paths with NeuralForecast

**Module 03: Sample Paths — The Correct Uncertainty Framework**

**Time:** ~15 minutes

## Learning Objectives

By the end of this notebook you will be able to:

1. Load and prepare the French Bakery dataset for neuralforecast
2. Train NHITS with `MQLoss` to obtain marginal quantile forecasts
3. Generate 100 sample paths using the `.simulate()` API
4. Visualize paths as an ensemble of plausible futures
5. Contrast sample path bounds with marginal quantile bounds on the same plot

## Prerequisites

- Guide 01: Sample Paths — The Correct Uncertainty Framework
- Guide 02: The Gaussian Copula Method
- `pip install neuralforecast datasetsforecast utilsforecast`

## 1. Imports and Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss

# Reproducibility
np.random.seed(42)

print("Imports complete.")

## 2. Load French Bakery Data

The French Bakery dataset contains daily sales of bakery items. We use baguette sales — a high-volume staple with clear weekly seasonality and day-to-day autocorrelation.

NeuralForecast expects a DataFrame with three columns:
- `unique_id`: series identifier (string)
- `ds`: date column (datetime)
- `y`: target variable (numeric)

In [ ]:
# Load the French Bakery dataset
url = (
    "https://raw.githubusercontent.com/Nixtla/transfer-learning-time-series"
    "/main/datasets/french_bakery.csv"
)
df_raw = pd.read_csv(url, parse_dates=["date"])

# Filter to baguette and rename columns to neuralforecast convention
df = (
    df_raw[df_raw["item"] == "BAGUETTE"][["date", "quantity"]]
    .copy()
    .rename(columns={"date": "ds", "quantity": "y"})
    .assign(unique_id="baguette")
    [["unique_id", "ds", "y"]]
    .sort_values("ds")
    .reset_index(drop=True)
)

print(f"Dataset: {len(df)} daily observations")
print(f"Date range: {df.ds.min().date()} to {df.ds.max().date()}")
print(f"\nBaguette sales — summary statistics:")
print(df["y"].describe().round(1))
df.head()

In [ ]:
# Train / test split: hold out the final 7 days
# h=7 means we forecast one week ahead — the test set is exactly one forecast horizon
H = 7
df_train = df.iloc[:-H].copy()
df_test = df.iloc[-H:].copy()

print(f"Training: {len(df_train)} days ({df_train.ds.min().date()} to {df_train.ds.max().date()})")
print(f"Test:     {len(df_test)} days ({df_test.ds.min().date()} to {df_test.ds.max().date()})")

# Quick look at the training series
fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(df_train["ds"], df_train["y"], linewidth=0.8, color="steelblue", alpha=0.85)
ax.set_title("French Bakery — Baguette Daily Sales (training data)")
ax.set_xlabel("Date")
ax.set_ylabel("Baguettes sold")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 3. Train NHITS with MQLoss

We train NHITS (Neural Hierarchical Interpolation for Time Series) with `MQLoss`. The key hyperparameters:

- `h=7`: forecast horizon — one full week
- `input_size=28`: four weeks of look-back history
- `MQLoss(level=[80, 90])`: request 80th and 90th percentile intervals. Internally this trains quantiles at `[0.05, 0.10, 0.20, ..., 0.80, 0.90, 0.95]`
- `scaler_type="robust"`: use median/IQR scaling, robust to daily outliers

Training takes ~1-2 minutes on CPU.

In [ ]:
model = NHITS(
    h=7,
    input_size=28,
    loss=MQLoss(level=[80, 90]),
    scaler_type="robust",
    max_steps=500,
)

nf = NeuralForecast(models=[model], freq="D")
nf.fit(df_train)

print("Training complete.")

In [ ]:
# Inspect the marginal quantile forecasts for the test horizon
forecasts = nf.predict()

print("Marginal quantile forecasts (one row per horizon step):")
print(forecasts.to_string(index=False))

The forecast DataFrame gives marginal quantile estimates — each row is one day, each column is one quantile level. These are the step-by-step distributions that the Gaussian Copula will stitch into correlated sample paths.

## 4. Generate 100 Sample Paths via `.simulate()`

The `.simulate()` method runs the full Gaussian Copula pipeline internally:

1. Uses the marginal quantile forecasts from Step 3
2. Estimates AR(1) autocorrelation from the training residuals
3. Builds the Toeplitz correlation matrix
4. Generates correlated uniform draws via Cholesky
5. Maps back to forecast scale via inverse quantile transform

The result: 100 internally consistent 7-day demand trajectories.

In [ ]:
# Generate 100 sample paths
N_PATHS = 100
paths_df = nf.models[0].simulate(n_paths=N_PATHS)

print(f"paths_df shape: {paths_df.shape}")
print(f"Columns: {paths_df.columns[:5].tolist()} ... {paths_df.columns[-2:].tolist()}")
print(f"\nFirst 3 paths for each horizon step:")
print(paths_df[["ds", "sample_1", "sample_2", "sample_3"]].to_string(index=False))

In [ ]:
# Extract paths as a numpy array: shape (n_paths, horizon)
# Convention: rows = paths, columns = horizon steps
path_cols = [c for c in paths_df.columns if c.startswith("sample_")]
paths = paths_df[path_cols].values.T   # (100, 7)

print(f"paths array shape: {paths.shape}")
print(f"  rows = individual paths ({paths.shape[0]})")
print(f"  cols = horizon steps ({paths.shape[1]})")

# Basic statistics across paths
print(f"\nMin demand in any path:  {paths.min():.0f}")
print(f"Max demand in any path:  {paths.max():.0f}")
print(f"Mean weekly total:       {paths.sum(axis=1).mean():.0f}")

## 5. Visualise the Sample Paths

Each thin line is one sample path — a complete plausible week of demand. The navy line is the median across all paths. The shaded band covers the central 80% of paths.

In [ ]:
horizon_dates = pd.to_datetime(paths_df["ds"].values)
horizon_days = np.arange(1, H + 1)

fig, ax = plt.subplots(figsize=(10, 5))

# Plot all 100 paths as thin, semi-transparent lines
for s in range(N_PATHS):
    ax.plot(horizon_days, paths[s], color="steelblue", alpha=0.10, linewidth=0.9)

# Overlay median and 80% band derived from the paths
median_path = np.median(paths, axis=0)
lo_80 = np.quantile(paths, 0.10, axis=0)
hi_80 = np.quantile(paths, 0.90, axis=0)

ax.fill_between(horizon_days, lo_80, hi_80,
                alpha=0.25, color="steelblue", label="80% band (from paths)")
ax.plot(horizon_days, median_path, color="navy", linewidth=2.5,
        label="Median path", zorder=5)

# Overlay actual test values
ax.plot(horizon_days, df_test["y"].values, color="crimson", linewidth=2,
        marker="o", markersize=5, label="Actual demand", zorder=6)

ax.set_title("100 Sample Paths — Baguette Demand, Next 7 Days", fontsize=13)
ax.set_xlabel("Forecast day")
ax.set_ylabel("Baguettes sold")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("sample_paths_baguette.png", dpi=150, bbox_inches="tight")
plt.show()

print("Figure saved: sample_paths_baguette.png")

## 6. Sample Path Bounds vs. Marginal Quantile Bounds

Both plots show an 80% uncertainty band. The bands look visually similar — but they encode very different information.

- **Left (sample path bounds):** The band comes from the 10th and 90th percentiles of the paths at each step. Because paths are correlated across steps, the band reflects the joint uncertainty.
- **Right (marginal quantile bounds):** The band comes directly from the model's `NHITS-lo-90` and `NHITS-hi-80` columns. Each step's band is computed independently.

For single-step questions, both bands give equivalent answers. For any multi-step question — weekly totals, reorder timing, cumulative risk — only the sample path band is correct.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax in axes:
    ax.plot(horizon_days, df_test["y"].values, color="crimson", linewidth=2,
            marker="o", markersize=5, label="Actual", zorder=6)
    ax.plot(horizon_days, median_path, color="navy", linewidth=2.5, label="Median")
    ax.set_xlabel("Forecast day")
    ax.grid(alpha=0.3)

# Left: sample path band
ax = axes[0]
for s in range(N_PATHS):
    ax.plot(horizon_days, paths[s], color="steelblue", alpha=0.07, linewidth=0.8)
ax.fill_between(horizon_days, lo_80, hi_80,
                alpha=0.30, color="steelblue", label="80% band (paths)")
ax.set_title("Sample Path Bounds\n(joint distribution)", fontsize=12)
ax.set_ylabel("Baguettes sold")
ax.legend(fontsize=9)

# Right: marginal quantile band
ax = axes[1]
marg_lo = forecasts["NHITS-lo-90"].values  # 10th percentile
marg_hi = forecasts["NHITS-hi-90"].values  # 90th percentile
ax.fill_between(horizon_days, marg_lo, marg_hi,
                alpha=0.30, color="orange", label="80% band (marginal)")
ax.set_title("Marginal Quantile Bounds\n(independent per step)", fontsize=12)
ax.legend(fontsize=9)

plt.suptitle(
    "Same model, different information content\n"
    "The bands look similar — their multi-step properties are not",
    fontsize=11
)
plt.tight_layout()
plt.savefig("paths_vs_marginals.png", dpi=150, bbox_inches="tight")
plt.show()

print("Figure saved: paths_vs_marginals.png")

## 7. Quantifying the Difference

The visual comparison shows the bands look similar. Here we show numerically that they encode different information for the multi-step question: "What is the 80th percentile of **weekly total** demand?"

In [ ]:
# Correct answer: 80th percentile of weekly totals across paths
weekly_totals = paths.sum(axis=1)   # sum each path
correct_80th = np.quantile(weekly_totals, 0.80)

# Naive answer: sum of marginal 90th percentiles (NHITS-hi-90 = 90th pct)
# This gives the 90th pct for each step; summing gives the wrong answer
naive_80th = forecasts["NHITS-hi-80"].sum()

print("=" * 50)
print("Weekly total demand — 80th percentile")
print("=" * 50)
print(f"  From sample paths (correct):       {correct_80th:.0f} baguettes")
print(f"  From marginal quantile sum (naive): {naive_80th:.0f} baguettes")
print(f"  Difference:                         {naive_80th - correct_80th:+.0f} baguettes")
print()
print("Actual test week total:", df_test['y'].sum())
print()
print("Distribution of simulated weekly totals:")
for pct in [10, 25, 50, 75, 80, 90]:
    print(f"  {pct:3d}th percentile: {np.quantile(weekly_totals, pct/100):.0f}")

The two methods give different answers for the weekly total. Only the sample path approach is correct, because it accounts for the temporal correlation across days. Notebook 02 builds on this to answer three concrete business decision problems.

## Summary

| Step | Code | Result |
|------|------|--------|
| Load data | `pd.read_csv(url)` | 3-column DataFrame (unique_id, ds, y) |
| Train model | `NHITS(h=7, loss=MQLoss(...))` | Marginal quantile forecasts |
| Generate paths | `nf.models[0].simulate(n_paths=100)` | (7, 102) DataFrame |
| Extract array | `paths_df[path_cols].values.T` | (100, 7) numpy array |
| Answer question | `np.quantile(paths.sum(axis=1), 0.80)` | 80th pct of weekly total |

## Next Steps

Notebook 02 applies the sample path array to three business decision problems: weekly totals, worst-case single day, and reorder timing.